# Attention + Residual Rollout for Visual Token Scoring

This notebook runs a dense LLaVA prefill pass, collects language-model self-attention maps, and approximates each original image token's influence on later text tokens with an attention-plus-residual provenance rollout.

In [ ]:
# Uncomment if Colab does not already have the needed packages.
# !pip install -q transformers accelerate bitsandbytes pillow requests

In [ ]:
import io
import requests
import torch
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
PROMPT = "USER: <image>\nWhat is in this image?\nASSISTANT:"
IMAGE_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
ALPHA = 0.7
LAYER_SLICE = slice(0, 4)  # earlier layers are usually less mixed
TOP_K = 32

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    attn_implementation="eager",  # needed for output_attentions=True
)
model.eval()

In [ ]:
def load_image(image_url: str) -> Image.Image:
    response = requests.get(image_url)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")


def get_dense_multimodal_inputs(model, processor, image, prompt):
    inputs = processor(text=prompt, images=image, return_tensors="pt")
    for key, value in inputs.items():
        if hasattr(value, "to"):
            inputs[key] = value.to(model.device)

    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask")

    text_embeds = model.get_input_embeddings()(input_ids)
    image_outputs = model.model.get_image_features(
        pixel_values=inputs["pixel_values"],
        vision_feature_layer=None,
        vision_feature_select_strategy=None,
        image_sizes=inputs.get("image_sizes"),
        return_dict=True,
    )

    image_features = image_outputs.pooler_output
    image_features = torch.cat(image_features, dim=0).to(text_embeds.device, text_embeds.dtype)
    if image_features.ndim == 2:
        image_features = image_features.unsqueeze(0)

    return inputs, input_ids, attention_mask, text_embeds, image_features


def merge_image_features_single_example(model, input_ids, inputs_embeds, attention_mask, image_features):
    image_token_id = model.config.image_token_id
    input_ids_1 = input_ids[0]
    embeds_1 = inputs_embeds[0]
    attn_1 = attention_mask[0] if attention_mask is not None else None
    device = input_ids.device

    image_positions = torch.where(input_ids_1 == image_token_id)[0]
    if image_positions.numel() == 0:
        raise ValueError("No <image> placeholder tokens found.")

    start = image_positions[0].item()
    end = image_positions[-1].item() + 1
    expected = torch.arange(start, end, device=device)
    if not torch.equal(image_positions, expected):
        raise ValueError("This notebook expects the image placeholder block to be contiguous.")

    prefix_embeds = embeds_1[:start, :]
    suffix_embeds = embeds_1[end:, :]
    merged_embeds = torch.cat([prefix_embeds, image_features[0], suffix_embeds], dim=0).unsqueeze(0)

    merged_input_ids = torch.cat(
        [input_ids_1[:start], torch.full((image_features.shape[1],), image_token_id, dtype=input_ids_1.dtype, device=device), input_ids_1[end:]],
        dim=0,
    ).unsqueeze(0)

    if attn_1 is None:
        merged_attention_mask = None
    else:
        merged_attention_mask = torch.cat(
            [attn_1[:start], torch.ones(image_features.shape[1], dtype=attn_1.dtype, device=device), attn_1[end:]],
            dim=0,
        ).unsqueeze(0)

    merged_position_ids = torch.arange(merged_embeds.shape[1], device=device).unsqueeze(0)
    return merged_input_ids, merged_embeds, merged_attention_mask, merged_position_ids


def rollout_visual_provenance(attentions, image_mask, text_mask, alpha=0.7, layer_slice=slice(0, 4)):
    selected_layers = attentions[layer_slice]
    seq_len = image_mask.numel()
    num_image_tokens = int(image_mask.sum().item())
    image_positions = torch.where(image_mask)[0]

    provenance = torch.zeros(seq_len, num_image_tokens, device=selected_layers[0].device, dtype=selected_layers[0].dtype)
    provenance[image_positions, torch.arange(num_image_tokens, device=provenance.device)] = 1.0

    for layer_attn in selected_layers:
        # [1, heads, q_len, k_len] -> [q_len, k_len]
        attn = layer_attn[0].mean(dim=0)
        provenance = alpha * provenance + (1.0 - alpha) * (attn @ provenance)

    image_scores = provenance[text_mask].sum(dim=0)
    return image_scores, provenance


In [ ]:
image = load_image(IMAGE_URL)
inputs, input_ids, attention_mask, text_embeds, image_features = get_dense_multimodal_inputs(
    model, processor, image, PROMPT
)
merged_input_ids, merged_embeds, merged_attention_mask, merged_position_ids = merge_image_features_single_example(
    model, input_ids, text_embeds, attention_mask, image_features
)

with torch.no_grad():
    outputs = model.model.language_model(
        inputs_embeds=merged_embeds,
        attention_mask=merged_attention_mask,
        position_ids=merged_position_ids,
        use_cache=False,
        output_attentions=True,
        return_dict=True,
    )

image_mask = merged_input_ids[0] == model.config.image_token_id
text_mask = ~image_mask
scores, provenance = rollout_visual_provenance(
    outputs.attentions,
    image_mask=image_mask,
    text_mask=text_mask,
    alpha=ALPHA,
    layer_slice=LAYER_SLICE,
)

topk = min(TOP_K, scores.numel())
top_scores, top_indices = torch.topk(scores, k=topk)

print("num layers with attentions:", len(outputs.attentions))
print("last attention shape:", tuple(outputs.attentions[-1].shape))
print("num image tokens:", int(image_mask.sum().item()))
print("top rollout-scored image tokens:")
for rank, (idx, score) in enumerate(zip(top_indices.tolist(), top_scores.tolist()), start=1):
    print(f"{rank:2d}. token {idx:3d} -> {score:.6f}")

Notes:

- This rollout is intentionally approximate. It models information flow through attention plus a fixed residual retention factor `alpha`.
- Earlier layers are usually less multimodally mixed, so `slice(0, 4)` is a reasonable starting point.
- To turn the scores into pruning, keep the top-k image token indices and rebuild the multimodal sequence with only those tokens.